In [ ]:
!pip install split-folders

import splitfolders
import os

pasta_descompactada = '/content/dataset_lixo'
pasta_classes = pasta_descompactada

for root, dirs, files in os.walk(pasta_descompactada):
    if 'glass' in dirs and 'metal' in dirs:
        pasta_classes = root
        break

print(f"Lendo imagens da pasta: {pasta_classes}")

splitfolders.ratio(
    pasta_classes,
    output="/content/dataset_yolo",
    seed=42,
    ratio=(0.8, 0.2) # 80% para treino, 20% para validação
)
print("Dataset organizado com sucesso!")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')

model.train(
  data='/content/dataset_yolo',
  epochs=50,
  imgsz=224,
  device=0
)

Testar o modelo treinado

In [ ]:
from ultralytics import YOLO
from google.colab import files
import IPython.display as display
import os

caminho_do_modelo = '/content/runs/classify/train/weights/best.pt'
modelo_treinado = YOLO(caminho_do_modelo)

print("Faça o upload de uma imagem para testar o modelo:")
uploaded = files.upload()

for nome_arquivo in uploaded.keys():
    caminho_imagem = f"/content/{nome_arquivo}"

    print(f"\nAnalisando a imagem: {nome_arquivo}...")

    resultados = modelo_treinado.predict(source=caminho_imagem, save=True)

    for resultado in resultados:
        indice_maior_chance = resultado.probs.top1
        nome_da_classe = resultado.names[indice_maior_chance]
        certeza = resultado.probs.top1conf.item() * 100

        print(f"\n--- RESULTADO ---")
        print(f"O modelo acha que isso é: {nome_da_classe.upper()}")
        print(f"Certeza de: {certeza:.2f}%\n")